# Task 6: House Price Prediction
**Internship:** DevelopersHub Corporation — AI/ML Engineering  
**Objective:** Predict house prices based on property features using regression models.  
**Dataset:** California Housing Dataset (via `sklearn`) — real-world housing data  
**Models:** Linear Regression & Gradient Boosting Regressor

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.dpi"] = 120
print("Libraries imported successfully.")

## 2. Load and Inspect the Dataset

In [ ]:
# Load California Housing Dataset
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()

# Target: MedHouseVal = Median house value (in $100,000s)
print("Shape:", df.shape)
print("\nFeature Descriptions:")
for name, desc in zip(housing.feature_names, housing.feature_names):
    print(f"  {name}")
print("  MedHouseVal (Target): Median house value in $100k")
print("\nFirst 5 rows:")
df.head()

In [ ]:
print("=== Dataset Info ===")
df.info()

In [ ]:
print("=== Descriptive Statistics ===")
df.describe().round(3)

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["MedHouseVal"], bins=50, color="#66c2a5", edgecolor="white")
axes[0].set_title("Distribution of House Prices (Target)", fontsize=12)
axes[0].set_xlabel("Median House Value ($100k)")
axes[0].set_ylabel("Frequency")

# Log-transformed target
axes[1].hist(np.log1p(df["MedHouseVal"]), bins=50, color="#fc8d62", edgecolor="white")
axes[1].set_title("Log-Transformed House Prices", fontsize=12)
axes[1].set_xlabel("Log(Median House Value + 1)")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.savefig("task6_target_dist.png")
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(10, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            linewidths=0.5, ax=ax, annot_kws={"size": 9})
ax.set_title("Feature Correlation Heatmap", fontsize=13)
plt.tight_layout()
plt.savefig("task6_correlation.png")
plt.show()

In [ ]:
# Scatter plots: key features vs target
key_features = ["MedInc", "AveRooms", "HouseAge", "AveOccup"]

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes = axes.flatten()

for ax, feat in zip(axes, key_features):
    # Sample for faster plotting
    sample = df.sample(2000, random_state=42)
    ax.scatter(sample[feat], sample["MedHouseVal"], alpha=0.3, s=8, color="#8da0cb")
    ax.set_xlabel(feat)
    ax.set_ylabel("Median House Value ($100k)")
    ax.set_title(f"{feat} vs House Price", fontsize=11)

plt.suptitle("Feature Relationships with House Price", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("task6_scatter_features.png", bbox_inches="tight")
plt.show()

In [ ]:
# Geographic distribution of house prices (using Latitude/Longitude)
fig, ax = plt.subplots(figsize=(9, 7))
scatter = ax.scatter(df["Longitude"], df["Latitude"],
                     c=df["MedHouseVal"], cmap="RdYlGn",
                     alpha=0.4, s=2)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("Median House Value ($100k)")
ax.set_title("Geographic Distribution of House Prices in California", fontsize=12)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.savefig("task6_geo_map.png")
plt.show()

## 4. Preprocessing — Feature Engineering & Scaling

In [ ]:
# Features and target
FEATURES = ["MedInc", "HouseAge", "AveRooms", "AveBedrms",
            "Population", "AveOccup", "Latitude", "Longitude"]
TARGET = "MedHouseVal"

X = df[FEATURES]
y = df[TARGET]

# Check for missing values
print("Missing values:", X.isnull().sum().sum())

# Train/test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"\nTraining samples: {X_train.shape[0]}")
print(f"Test samples:     {X_test.shape[0]}")

## 5. Model Training

### 5a. Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)

mae_lr  = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr   = r2_score(y_test, y_pred_lr)

print("=== Linear Regression ===")
print(f"MAE:  {mae_lr:.4f}")
print(f"RMSE: {rmse_lr:.4f}")
print(f"R²:   {r2_lr:.4f}")

### 5b. Gradient Boosting Regressor

In [ ]:
gbr = GradientBoostingRegressor(
    n_estimators=200, max_depth=4, learning_rate=0.1,
    subsample=0.8, random_state=42
)
gbr.fit(X_train_scaled, y_train)
y_pred_gbr = gbr.predict(X_test_scaled)

mae_gbr  = mean_absolute_error(y_test, y_pred_gbr)
rmse_gbr = np.sqrt(mean_squared_error(y_test, y_pred_gbr))
r2_gbr   = r2_score(y_test, y_pred_gbr)

print("=== Gradient Boosting Regressor ===")
print(f"MAE:  {mae_gbr:.4f}")
print(f"RMSE: {rmse_gbr:.4f}")
print(f"R²:   {r2_gbr:.4f}")

## 6. Model Evaluation & Visualization

### 6a. Model Comparison Table

In [ ]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Gradient Boosting"],
    "MAE":   [round(mae_lr, 4),   round(mae_gbr, 4)],
    "RMSE":  [round(rmse_lr, 4),  round(rmse_gbr, 4)],
    "R²":    [round(r2_lr, 4),    round(r2_gbr, 4)]
})
print("=== Model Performance Comparison ===")
print(results.to_string(index=False))

### 6b. Actual vs Predicted Prices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, y_pred, title, color in zip(
    axes,
    [y_pred_lr, y_pred_gbr],
    ["Linear Regression", "Gradient Boosting"],
    ["#66c2a5", "#fc8d62"]
):
    ax.scatter(y_test, y_pred, alpha=0.25, s=5, color=color)
    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], "k--", linewidth=1.5, label="Perfect Prediction")
    ax.set_title(f"Actual vs Predicted — {title}", fontsize=11)
    ax.set_xlabel("Actual Price ($100k)")
    ax.set_ylabel("Predicted Price ($100k)")
    ax.legend(fontsize=8)

plt.suptitle("Actual vs Predicted House Prices", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("task6_actual_vs_predicted.png", bbox_inches="tight")
plt.show()

### 6c. Residual Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_pred, title, color in zip(
    axes,
    [y_pred_lr, y_pred_gbr],
    ["Linear Regression", "Gradient Boosting"],
    ["#66c2a5", "#fc8d62"]
):
    residuals = y_test - y_pred
    ax.scatter(y_pred, residuals, alpha=0.25, s=5, color=color)
    ax.axhline(0, color="black", linewidth=1.2, linestyle="--")
    ax.set_title(f"Residual Plot — {title}", fontsize=11)
    ax.set_xlabel("Predicted Price ($100k)")
    ax.set_ylabel("Residuals")

plt.suptitle("Residual Analysis", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("task6_residuals.png", bbox_inches="tight")
plt.show()

### 6d. Feature Importance (Gradient Boosting)

In [ ]:
importances = gbr.feature_importances_
sorted_idx = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(range(len(FEATURES)),
       importances[sorted_idx],
       color=sns.color_palette("Set2", len(FEATURES)))
ax.set_xticks(range(len(FEATURES)))
ax.set_xticklabels([FEATURES[i] for i in sorted_idx], rotation=30, ha="right")
ax.set_title("Feature Importances — Gradient Boosting", fontsize=13)
ax.set_ylabel("Importance Score")
plt.tight_layout()
plt.savefig("task6_feature_importance.png")
plt.show()

print("\nFeature Importances (Ranked):")
for i in sorted_idx:
    print(f"  {FEATURES[i]:<15} {importances[i]:.4f}")

## 7. Key Insights & Findings

1. **Gradient Boosting significantly outperforms Linear Regression** in all metrics (lower MAE/RMSE, higher R²), confirming non-linear relationships in housing data.
2. **MedInc (Median Income)** is by far the most important predictor — wealthier areas consistently have higher home values.
3. **Latitude and Longitude** rank highly, capturing geographic pricing patterns (coastal vs. inland California).
4. **Linear Regression** still achieves reasonable performance as a baseline and is much faster to train.
5. **Residual plots** show Linear Regression underpredicts very high-value homes, while Gradient Boosting residuals are more centered around zero.
6. The **geographic scatter plot** clearly shows Bay Area and LA coastal regions commanding premium prices.

---
*Task 6 Complete — DevelopersHub AI/ML Internship*